# Data Acquisition and Management

This notebook contains notes, examples, and exercises for Module 2 of the Data Science Fundamentals course.

## Learning Objectives
- Understand different data sources.
- Read and write data from flat files.
- Work with APIs.
- Scrape data from web pages.
- Connect to SQL databases.
- Manage large datasets efficiently.
- Understand basic data pipelines and versioning.

## Why Data Acquisition Matters

Data acquisition is the process of collecting and preparing data from various sources. It's often the most time-consuming part of a data science project. Without high-quality data, analysis and modeling will be flawed.

## Reading Flat Files

We start by reading common file types such as CSV, Excel, JSON, and Parquet.

In [1]:
import pandas as pd

# CSV
# df_csv = pd.read_csv('data.csv')

# Excel
# df_excel = pd.read_excel('data.xlsx')

# JSON
# df_json = pd.read_json('data.json')

# Parquet
# df_parquet = pd.read_parquet('data.parquet')

## Accessing APIs

APIs provide programmatic access to data. Here's how to access a simple API and read the JSON response.

In [9]:
#example 1
import requests

url = 'https://catfact.ninja/fact'
response = requests.get(url)
data = response.json()

print("Random cat fact:", data['fact'])

Random cat fact: Some notable people who disliked cats:  Napoleon Bonaparte, Dwight D. Eisenhower, Hitler.


In [10]:
#example 2
import requests

url = 'https://dog.ceo/api/breeds/image/random'
response = requests.get(url)
data = response.json()

print("Random dog image URL:", data['message'])

Random dog image URL: https://images.dog.ceo/breeds/schnauzer-miniature/n02097047_151.jpg


## Web Scraping

Use BeautifulSoup to extract data from web pages. Use responsibly!

In [11]:
from bs4 import BeautifulSoup

html = requests.get('https://books.toscrape.com/').text
soup = BeautifulSoup(html, 'html.parser')
titles = [book.h3.a['title'] for book in soup.select('.product_pod')]
titles[:5]

['A Light in the Attic',
 'Tipping the Velvet',
 'Soumission',
 'Sharp Objects',
 'Sapiens: A Brief History of Humankind']

## SQL Databases

Use SQLite for local testing. In production, you'd use PostgreSQL, MySQL, or similar.

In [12]:
import sqlite3

conn = sqlite3.connect('example.db')
df_sql = pd.read_sql_query('SELECT name FROM sqlite_master WHERE type="table";', conn)
df_sql

,name


## Large Dataset Handling

Read data in chunks or optimize data types to handle large files.

In [15]:
# for chunk in pd.read_csv('./data/forestfires.csv', chunksize=10000):
#     process(chunk)


total_temp = 0
total_rows = 0

for chunk in pd.read_csv('./data/forestfires.csv', chunksize=1000):
    total_temp += chunk['temp'].sum()
    total_rows += len(chunk)

average_temp = total_temp / total_rows
print("Average temperature:", average_temp)

Average temperature: 18.88916827852998


## Data Pipelines & Versioning

###  What is Data Versioning?
Just like code, data versioning is about tracking changes in datasets over time. It’s especially important when you:

- Train models on specific snapshots of data.

- Need to audit or reproduce past analyses.

- Collaborate with others who might modify datasets.

Without versioning, it’s easy to lose track of which data was used, making it hard to debug issues or understand how results evolved.

### How to Version Data?
- Manual versioning:

Save copies of raw and processed data with clear timestamps or version numbers in filenames.

Example: sales_raw_2025_06_29.csv, sales_clean_v2.csv.

- Automated version control:

Tools like DVC (Data Version Control) integrate with Git and track data and models efficiently.

Cloud storage with metadata tagging (e.g., AWS S3 versioning).

- Benefits of Versioning
Enables rollback to previous datasets if errors are found.

Facilitates collaboration by sharing specific data versions.

Enhances transparency and reproducibility in data projects.

In [21]:
### 1. Manual Data Versioning with Timestamps in Filenames

import pandas as pd
from datetime import datetime

# Example data (usually loaded or processed)
data = {
    'user_id': [1, 2, 3],
    'activity': ['login', 'purchase', 'logout'],
    'timestamp': ['2025-06-29 10:00', '2025-06-29 10:05', '2025-06-29 10:15']
}

df = pd.DataFrame(data)

# Create a versioned filename using current datetime
version = datetime.now().strftime("%Y%m%d_%H%M%S")
filename = f"./data/user_activity_v{version}.csv"

# Save CSV with version in filename
df.to_csv(filename, index=False)

print(f"Data saved as: {filename}")

Data saved as: ./data/user_activity_v20250629_192142.csv


In [23]:
### 2. Load a Specific Version

# Load a specific version
version_to_load = "20250629_192142"  # example version string
filename = f"./data/user_activity_v{version_to_load}.csv"

df_versioned = pd.read_csv(filename)
print(df_versioned.head())

   user_id  activity         timestamp
0        1     login  2025-06-29 10:00
1        2  purchase  2025-06-29 10:05
2        3    logout  2025-06-29 10:15


In [24]:
### Simple Versioning with Incremental Version Numbers

import os

def get_next_version(version_file="version.txt"):
    if os.path.exists(version_file):
        with open(version_file, "r") as f:
            version = int(f.read().strip())
        version += 1
    else:
        version = 1
    with open(version_file, "w") as f:
        f.write(str(version))
    return version

version = get_next_version()
filename = f"./data/user_activity_v{version}.csv"

# Save data
df.to_csv(filename, index=False)
print(f"Data saved as: {filename}")

Data saved as: ./data/user_activity_v2.csv


In [25]:
### 4. Conceptual Example: Using DVC (Data Version Control)
#DVC is a tool that tracks data files and models along with your code using Git. 
#While it requires command-line use, here’s how you’d conceptually version a dataset with DVC:

# Initialize DVC repo (run once)
dvc init

# Add dataset to DVC tracking
dvc add user_activity.csv

# Commit changes with Git
git add user_activity.csv.dvc .gitignore
git commit -m "Add versioned user activity data"

# Later, update the dataset and track new version
dvc add user_activity.csv
git add user_activity.csv.dvc
git commit -m "Update user activity data"

SyntaxError: invalid syntax (499548321.py, line 6)